In [20]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)


In [21]:
df=pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn (1).csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [23]:
df[["tenure", "MonthlyCharges"]].describe()


,tenure,MonthlyCharges
count,7043.000000,7043.000000
mean,32.371149,64.761692
std,24.559481,30.090047
min,0.000000,18.250000
25%,9.000000,35.500000
50%,29.000000,70.350000
75%,55.000000,89.850000
max,72.000000,118.750000


In [24]:
df["Contract"].value_counts()


Month-to-month    3875
Two year          1695
One year          1473
Name: Contract, dtype: int64

In [25]:
df["PaymentMethod"].value_counts()


Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: PaymentMethod, dtype: int64

In [26]:
df["Churn"].value_counts()


No     5174
Yes    1869
Name: Churn, dtype: int64

In [27]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)



In [28]:
encoder = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    if col != "customerID":
        df[col] = encoder.fit_transform(df[col])


In [29]:
X = df.drop(["customerID", "Churn"], axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [30]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)


DecisionTreeClassifier(random_state=42)

In [31]:
y_pred_dt = dt_model.predict(X_test)

accuracy_dt = accuracy_score(y_test, y_pred_dt)
cm_dt = confusion_matrix(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
precision_df = precision_score(y_test, y_pred_dt)
accuracy_dt, cm_dt, recall_dt, f1_dt, precision_df


(0.730305180979418,
 array([[835, 200],
        [180, 194]], dtype=int64),
 0.5187165775401069,
 0.5052083333333334,
 0.49238578680203043)

In [32]:
tn, fp, fn, tp = cm_dt.ravel()

print("Churn customers correctly identified (TP):", tp)
print("Loyal customers wrongly flagged (FP):", fp)


Churn customers correctly identified (TP): 194
Loyal customers wrongly flagged (FP): 200


In [33]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)



XGBClassifier(base_score=0.5, booster='gbtree', callbacks=None,
              colsample_bylevel=1, colsample_bynode=1, colsample_bytree=0.8,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', gamma=0, gpu_id=-1,
              grow_policy='depthwise', importance_type=None,
              interaction_constraints='', learning_rate=0.05, max_bin=256,
              max_cat_to_onehot=4, max_delta_step=0, max_depth=4, max_leaves=0,
              min_child_weight=1, missing=nan, monotone_constraints='()',
              n_estimators=200, n_jobs=0, num_parallel_tree=1, predictor='auto',
              random_state=42, reg_alpha=0, reg_lambda=1, ...)

In [34]:
y_pred_xgb = xgb_model.predict(X_test)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)

accuracy_xgb, precision_xgb, recall_xgb, f1_xgb


(0.8105039034776437,
 0.6877192982456141,
 0.5240641711229946,
 0.5948406676783005)

In [35]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
cm_xgb


array([[946,  89],
       [178, 196]], dtype=int64)

In [36]:
tn, fp, fn, tp = cm_xgb.ravel()

print("Churn customers correctly identified (TP):", tp)
print("Churn customers missed (FN):", fn)
print("Loyal customers wrongly flagged (FP):", fp)
print("Loyal customers correctly retained (TN):", tn)


Churn customers correctly identified (TP): 196
Churn customers missed (FN): 178
Loyal customers wrongly flagged (FP): 89
Loyal customers correctly retained (TN): 946


In [37]:
comparison = pd.DataFrame({
    "Model": ["Decision Tree", "XGBoost"],
    "Accuracy": [accuracy_dt, accuracy_xgb],
    "Precision": [precision_df, precision_xgb],
    "Recall (Churn)": [recall_dt, recall_xgb],
    "F1-Score": [f1_dt, f1_xgb]
})

comparison


,Model,Accuracy,Precision,Recall (Churn),F1-Score
0,Decision Tree,0.730305,0.492386,0.518717,0.505208
1,XGBoost,0.810504,0.687719,0.524064,0.594841


In [38]:
feature_importance_xgb = pd.Series(
    xgb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance_xgb.head(10)


Contract            0.337504
OnlineSecurity      0.114254
InternetService     0.094337
TechSupport         0.080417
tenure              0.047264
PaperlessBilling    0.045727
MonthlyCharges      0.032188
MultipleLines       0.027826
StreamingMovies     0.027663
TotalCharges        0.024201
dtype: float32